In [ ]:
import os
os.makedirs('./model', exist_ok=True)

In [ ]:
%%writefile teacher_trainer.py
import torch.nn as nn
import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader
from evaluation import mc_cal


def train_teacher(model, config, known_unknown=0, unknown_unknown=1):

    T = config['T']
    M = config['M']
    R = config['R']
    p = config['p']
    rw= config['rw']

    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010)),
    ])

    allowed_indiced = []
    for i in range(10):
        if i not in [known_unknown, unknown_unknown]:
            allowed_indiced.append(i)

    batch_size = 64
    learning_rate = 0.001
    num_epochs = 100
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)
    train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)

    def filter_digits(dataset, exclude):
        keep_indices = [i for i, (_, label) in enumerate(dataset) if label not in exclude]
        return Subset(dataset, keep_indices)

    train_dataset = filter_digits(train_dataset, exclude=[known_unknown, unknown_unknown])
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.1,              # initial learning rate
        momentum=0.9,        # helps smooth gradients
        weight_decay=5e-4     # important for regularization
    )

    schedule = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[40, 70],
        gamma=0.1
    )
    criterion = nn.CrossEntropyLoss()

    print("*** Start teacher training")
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for data, target in train_loader:
            index = torch.randint(0, M, [len(target)])
            
            data, target = data.to(device), target.to(device)

            for i in range(len(target)):
                target[i] = allowed_indiced.index(target[i])
            
            optimizer.zero_grad()
            output = model(data, index)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}")
        schedule.step()

    model.train()
    with torch.no_grad():
        result = mc_cal(model, test_loader, allowed_indiced, T=T, M=M, R=R)
        print(result)

    torch.save(model.state_dict(), "./model/teacher_model.pth")


In [ ]:
%%writefile teacher.py
import torch
import torch.nn as nn



class ControlledDropout(nn.Module):
    def __init__(self, p, shape, device, M=10, rw=.5):
        super(ControlledDropout, self).__init__()
        self.store = []
        for i in range(M):
            mask = torch.empty(shape, device=device).bernoulli_(1 - p)
            self.store.append(mask)

        self.rw = rw
        self.p = p

    def forward(self, x, index):
        selected = torch.stack([self.store[i] for i in index], dim=0)
        random_walk = torch.ones_like(selected, device=x.device) * self.rw
        random_walk = random_walk + selected * (2*self.p-1)/(1-self.p) * self.rw
        random_walk = torch.bernoulli(random_walk)

        mask = (selected.int() ^ random_walk.int()) / (1 - self.p)

        return x * mask

    def get_candidates(self, indices):
        return [self.store[i] for i in indices]




class VGG(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.device = cfg['device']
        self.p = cfg['p']
        self.M = cfg['M']
        self.rw = cfg['rw']

        self.features = self._make_layers(cfg['VGG16'])
        self.classifier = nn.Linear(256, cfg['num_classes'])

    def forward(self, x, indices):

        for layer in self.features:
            if isinstance(layer, ControlledDropout):
                x = layer(x, indices)
            else:
                x = layer(x)
        
        x = torch.flatten(x, 1)
        return self.classifier(x)

    def _make_layers(self, cfg):
        layers = []
        in_channels = 3
        h, w = 32, 32  # CIFAR-10 input size

        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(2, 2))
                h //= 2
                w //= 2

            elif v == 'P':
                shape = (in_channels, h, w)
                layers.append(
                    ControlledDropout(
                        p=self.p,
                        shape=shape,
                        device=self.device,
                        M=self.M,
                        rw=self.rw
                    )
                )

            else:
                layers.extend([
                    nn.Conv2d(in_channels, v, kernel_size=3, padding=1, bias=False),
                    nn.BatchNorm2d(v),
                    nn.ReLU(inplace=True)
                ])
                in_channels = v

        layers.append(nn.AdaptiveAvgPool2d((1, 1)))
        return nn.Sequential(*layers)

    
    def get_candidates(self, indices):
        masks = []
        for layer in self.features:
            if isinstance(layer, ControlledDropout):
                masks.append(layer.get_candidates(indices))

        return masks


def vgg16_cifar_teacher(config):
    return VGG(config)


In [ ]:
%%writefile student.py
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F



class SoftDropout(nn.Module):
    def __init__(self, p, LCMC=True):
        super(SoftDropout, self).__init__()

        self.agg = nn.Conv3d(3, 1, kernel_size=1, bias=True)
        self.p = p
        self.LCMC = LCMC

    def forward(self, x, masks):

        if not self.LCMC:
            return x

        masks = torch.stack(masks, dim=0).unsqueeze(0)
        mask = self.agg(masks)
        mask = F.sigmoid(mask.squeeze(dim=1)) / (1-self.p)

        return x * mask




class VGG(nn.Module):
    def __init__(self, cfg, num_classes=8):
        super().__init__()

        self.features = self._make_layers(cfg)

        self.fc2 = nn.Linear(256, num_classes)
        self.fc3 = nn.Linear(256, 64)
        self.fc4 = nn.Linear(64, 4)
        self.fc5 = nn.Linear(12, 12)
        self.fc6 = nn.Linear(12, 12)

        self.fc7 = nn.Linear(12, 1)

        self.LDMC = cfg['LDMC']
        self.p = cfg['p']
        

    def forward(self, x, masks=None):
        c_mask = 0
        for layer in self.features:
            if isinstance(layer, SoftDropout):
                x = layer(x, masks[c_mask])
                c_mask += 1
            else:
                x = layer(x)

        x = torch.flatten(x, 1)
        
        cls = self.fc2(x)

        x = F.relu(self.fc3(x))

        x = self.fc4(x)

        x = F.silu(torch.cat((x, cls), dim=-1))

        x = F.relu(self.fc5(x))
        x = F.relu(self.fc6(x))

        unc = F.softplus(self.fc7(x))

        return cls, unc


    def _make_layers(self, config):
        layers = []
        in_channels = 3
        h, w = 32, 32  # CIFAR-10 input size

        cfg = config['VGG16']
        
        for v in cfg:
            if v == 'M':
                layers.append(nn.MaxPool2d(2, 2))
                h //= 2
                w //= 2

            elif v == 'P':
                layers.append(
                    SoftDropout(config['p'], config['LDMC'])
                    # nn.Conv3d(3, 1, kernel_size=(1, 1, 1), bias=True)
                )

            else:
                layers.extend([
                    nn.Conv2d(in_channels, v, kernel_size=3, padding=1, bias=False),
                    nn.BatchNorm2d(v),
                    nn.ReLU(inplace=True)
                ])
                in_channels = v

        layers.append(nn.AdaptiveAvgPool2d((1, 1)))
        return nn.Sequential(*layers)


def vgg16_cifar_student(cfgs):
    return VGG(cfgs)




In [ ]:
%%writefile evaluation.py
import numpy as np
from torch import nn
import torch.nn.functional as F
import torch
from sklearn.metrics import confusion_matrix

def mc_cal(model, dataloader, allowed_indices, T=32, M=10 , R=3, num_classes=10):
    model.train()  # Keep dropout active
    device = next(model.parameters()).device

    # Initialize accumulators
    class_correct = np.zeros(num_classes)
    class_counts = np.zeros(num_classes)
    class_aleatoric = np.zeros(num_classes)
    class_total = np.zeros(num_classes)

    for data, target in dataloader:
        data = data.to(device)

        # Monte Carlo samples
        candidates = torch.randperm(M)[:R].to(device)
        probs_T = []

        for _ in range(T):
            index = candidates[torch.randint(0, R, [len(target)])]
            out = model(data, index)
            probs = F.softmax(out, dim=1)
            probs_T.append(probs.unsqueeze(0))  # [1, batch, num_classes]
        probs_T = torch.cat(probs_T, dim=0)  # [T, batch, num_classes]

        probs_T_s = probs_T / probs_T.sum(dim=-1, keepdim=True)
        mean_probs_s = probs_T_s.mean(dim=0)

        # Mean predictive distribution
        mean_probs = probs_T.mean(dim=0)  # [batch, num_classes]

        # --- Entropies ---

        aleatoric_all = -(probs_T_s.clamp(.00001) * probs_T_s.clamp(.00001).log2()).sum(dim=2)
        aleatoric = aleatoric_all.mean(dim=0)


        # Total (predictive) entropy
        total_ent = -(mean_probs_s.clamp(.00001) * mean_probs_s.clamp(.00001).log2()).sum(dim=1)  # [batch]

        preds = mean_probs.argmax(dim=1)

        for i in range(len(target)):
            cls = target[i].item()

            class_counts[cls] += 1
            class_aleatoric[cls] += aleatoric[i].item()
            class_total[cls] += total_ent[i].item()
            class_correct[cls] += (allowed_indices[preds[i]] == target[i]).item()

    # Compute averages
    avg_acc = class_correct / np.maximum(class_counts, 1)
    avg_aleatoric = class_aleatoric / np.maximum(class_counts, 1)
    avg_total = class_total / np.maximum(class_counts, 1)
    avg_epistemic = avg_total - avg_aleatoric

    return {
        "accuracy_per_class": avg_acc,
        "aleatoric_per_class": avg_aleatoric,
        "total_per_class": avg_total,
        "epistemic_per_class": avg_epistemic,
    }



def mc_dropout_cal_sample(model, batch, T=20, M=10, R=3):
    model.train()  # keep dropout active

    device = next(model.parameters()).device

    probs_T = []

    batch_size = batch.size(0)
    candidates = torch.randperm(M)[:R].to(device)
    for _ in range(T):
        index = candidates[torch.randint(0, R, [batch_size])]
        out = model(batch, index)
        probs = F.softmax(out, dim=1)
        probs_T.append(probs.unsqueeze(0))  # [1, batch, num_classes]
    probs_T = torch.cat(probs_T, dim=0)  # [T, batch, num_classes]

    masks = model.get_candidates(candidates)

    # Mean predictive distribution
    mean_probs = probs_T.mean(dim=0)  # [batch, num_classes]

    aleatoric_all = -(probs_T.clamp(.001) * probs_T.clamp(.001).log2()).sum(dim=2)
    aleatoric = aleatoric_all.mean(dim=0)


    total_ent = -(mean_probs.clamp(.00001) * mean_probs.clamp(.00001).log2()).sum(dim=-1)  # [batch]
    epistemic = total_ent - aleatoric


    return {
        "preds": mean_probs,
        "aleatoric": aleatoric,
        "epistemic": epistemic,
        "total": total_ent,
        "total-val": total_ent,
        "masks": masks
    }



def get_uncertainty_class(pred_entropy):
    boarders = [torch.log2(torch.tensor(1.2)), torch.log2(torch.tensor(1.8)), torch.log2(torch.tensor(3)), torch.log2(torch.tensor(5))]

    uncertainty = -torch.ones((len(pred_entropy), 4), dtype=torch.float)
    for j in range(len(pred_entropy)):
        dis = torch.tensor([(pred_entropy[j] - boarders[i]) for i in range(len(boarders))])
        uncertainty[j, :] = dis > 0

    return uncertainty



@torch.no_grad()
def evaluate_student_per_class(student, teacher, test_loader, device, T=20, num_classes=10, M=10, R=3, known_unknown=0, unknown_unknown=1):
    student.eval()
    teacher.train()

    allowed_indices = []
    for i in range(num_classes):
        if i not in [known_unknown, unknown_unknown]:
            allowed_indices.append(i)


    # Initialize accumulators
    class_correct = np.zeros(num_classes)
    class_total = np.zeros(num_classes)
    class_aleatoric = np.zeros(num_classes)
    class_aleatoric_avg = np.zeros(num_classes)
    class_total_ent = np.zeros(num_classes)
    class_kl_loss = np.zeros(num_classes)

    loss_func = nn.KLDivLoss(reduction='batchmean')

    all_unc = []
    all_labels = []

    for data, target in test_loader:
        data, target = data.to(device), target.to(device)

        # --- Teacher MC Dropout outputs ---
        detail = mc_dropout_cal_sample(teacher, data, T=T)
        teacher_total = detail["aleatoric"]


        cnadidates = torch.randperm(M)[:R].to(device)
        masks = teacher.get_candidates(cnadidates)

        student_output, student_unc = student(data, masks)
        preds = student_output.argmax(dim=1)

        student_pred = F.log_softmax(student_output, dim=1)


        for i in range(len(target)):
            cls = target[i].item()

            preds[i] = allowed_indices[preds[i]]

            class_total[cls] += 1
            class_correct[cls] += (preds[i] == target[i]).item()
            class_aleatoric[cls] += np.power(teacher_total[i].item() - student_unc[i].item(), 2)
            class_aleatoric_avg[cls] += student_unc[i].item()
            class_total_ent[cls] += teacher_total[i].item()
            class_kl_loss[cls] += loss_func(student_pred[i], detail['preds'][i])

            if target[i].item()==unknown_unknown:
                all_unc.append(student_unc[i].item())
                all_labels.append(teacher_total[i].item())

    # Compute averages
    acc_per_class = class_correct / np.maximum(class_total, 1)
    aleatoric_per_class = np.sqrt(class_aleatoric / np.maximum(class_total, 1))
    total_per_class = class_total_ent / np.maximum(class_total, 1)
    aleatoric_avg_per_class = class_aleatoric_avg / np.maximum(class_total, 1)
    class_kl_loss = class_kl_loss / np.maximum(class_total, 1)



    # Return dictionary
    return {
        "accuracy_per_class": acc_per_class,
        "aleatoric_per_class": aleatoric_per_class,
        "total_per_class": total_per_class,
        "aleatoric_avg_per_class": aleatoric_avg_per_class,
        "class_kl_loss": class_kl_loss
    }

In [ ]:
%%writefile main.py
import os
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import sys
from evaluation import mc_dropout_cal_sample, evaluate_student_per_class
from teacher_trainer import train_teacher
import torch.nn as nn
from student import vgg16_cifar_student
from student import SoftDropout
from teacher import vgg16_cifar_teacher


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def initialize_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)




config = {
    'VGG16': [32, 64, 'M',
              64, 64, 'M',
              128, 128, 128, 'M',
              256, 256, 256, 'M',
              'P',
              256, 256, 256, 'M',
              'P'],
    'num_classes': 8, 'p':0.3, 'rw':.4, 'M':10, 'T':128, 'R':3,
    'device': device,
    'LDMC': True
}


batch_size = 32
learning_rate = 0.001
num_epochs = 40

known_unknown, unknown_unknown = 2, 3



teacher = vgg16_cifar_teacher(config).to(device)

teacher.apply(initialize_weights)

train_teacher(teacher, config, known_unknown=known_unknown, unknown_unknown=unknown_unknown)
student = vgg16_cifar_student(config).to(device)


transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])


test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)
train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)

def filter_digits(dataset, exclude):
    keep_indices = [i for i, (_, label) in enumerate(dataset) if label not in exclude]
    return Subset(dataset, keep_indices)


train_dataset = filter_digits(train_dataset, exclude=[unknown_unknown])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


optimizer = torch.optim.SGD(
    student.parameters(),
    lr=0.1,  # initial learning rate
    momentum=0.9,  # helps smooth gradients
    weight_decay=5e-4  # important for regularization
)


scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    milestones=[20, 40, 60],
    gamma=0.1
)


criterion_unc = F.binary_cross_entropy_with_logits
criterion_cls = nn.KLDivLoss(reduction='batchmean')



# initialize the student
student.apply(initialize_weights)
state_dict = teacher.features.state_dict()

student.features.load_state_dict(state_dict, strict=False)
for m in student.features.modules():
    if not isinstance(m, SoftDropout):
        for p in m.parameters():
            p.requires_grad = False



gamma = .75

for epoch in range(num_epochs):
    student.train()
    total_loss = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        with torch.no_grad():
            detail = mc_dropout_cal_sample(teacher, data, T=config['T'])

        optimizer.zero_grad()
        output, unc = student(data, detail['masks'])
        output = F.log_softmax(output, dim=-1)
        loss_cls = criterion_cls(output, detail['preds'])
        loss_unc = F.mse_loss(unc.squeeze(), detail["aleatoric"])

        loss = loss_cls + gamma * loss_unc

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}")

    if (epoch+1) % 20 == 0:
        torch.save(student.state_dict(), "./model/student_model-1.pth")
        print(evaluate_student_per_class(student, teacher, train_loader, device, T=config['T'], num_classes=10,
                                        known_unknown=known_unknown, unknown_unknown=unknown_unknown,
                                        M=config['M'], R=config['R']))

    if (epoch+1) % 20 == 0:
        print(evaluate_student_per_class(student, teacher, test_loader, device, T=config['T'], num_classes=10,
                                        known_unknown=known_unknown, unknown_unknown=unknown_unknown,
                                        M=config['M'], R=config['R']))

print("*****************---------*****************")



In [ ]:
!python main.py